# 🧪 Testing & CI/CD for Bioinformatics

## Learning Objectives
- Write unit tests with pytest
- Test bioinformatics functions
- Set up continuous integration with GitHub Actions
- Automate quality checks

## Why Test Bioinformatics Code?

**Common bugs in bioinformatics:**
- Off-by-one errors (0 vs 1-based coordinates)
- Strand handling (+/-/unknown)
- Missing data (N, gaps, ambiguous bases)
- Edge cases (empty sequences, single residues)
- File format variations

**Testing catches these BEFORE publication.**

---
## Part 1: Writing Tests with pytest

In [ ]:
# Our bioinformatics module (bio_utils.py)
def gc_content(sequence):
    """Calculate GC content as percentage."""
    if not sequence:
        return 0.0
    seq = sequence.upper()
    gc = seq.count('G') + seq.count('C')
    return (gc / len(seq)) * 100

def reverse_complement(sequence):
    """Return reverse complement of DNA sequence."""
    complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}
    return ''.join(complement.get(b, 'N') for b in reversed(sequence.upper()))

def translate(dna, table=1):
    """Translate DNA to protein."""
    codons = {
        'ATG': 'M', 'TAA': '*', 'TAG': '*', 'TGA': '*',
        'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
        'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
        # ... (abbreviated)
    }
    protein = []
    for i in range(0, len(dna) - 2, 3):
        codon = dna[i:i+3].upper()
        aa = codons.get(codon, 'X')
        if aa == '*':
            break
        protein.append(aa)
    return ''.join(protein)

In [ ]:
# Test file: test_bio_utils.py
test_code = '''
import pytest
from bio_utils import gc_content, reverse_complement, translate

class TestGCContent:
    def test_basic(self):
        assert gc_content("ATGC") == 50.0
    
    def test_all_gc(self):
        assert gc_content("GGCC") == 100.0
    
    def test_no_gc(self):
        assert gc_content("AAAA") == 0.0
    
    def test_empty_sequence(self):
        assert gc_content("") == 0.0
    
    def test_lowercase(self):
        assert gc_content("atgc") == 50.0
    
    def test_with_n(self):
        # N should not count as GC
        assert gc_content("GCNN") == 50.0


class TestReverseComplement:
    def test_basic(self):
        assert reverse_complement("ATGC") == "GCAT"
    
    def test_palindrome(self):
        # GAATTC (EcoRI site) is palindromic
        assert reverse_complement("GAATTC") == "GAATTC"
    
    def test_handles_n(self):
        assert reverse_complement("ATNG") == "CNAT"


class TestTranslate:
    def test_start_codon(self):
        assert translate("ATG") == "M"
    
    def test_stop_codon(self):
        assert translate("ATGTAA") == "M"
    
    def test_incomplete_codon(self):
        # Last 2 bases should be ignored
        assert translate("ATGAT") == "M"


# Parametrized tests (many inputs, same logic)
@pytest.mark.parametrize("seq,expected", [
    ("GGGG", 100.0),
    ("CCCC", 100.0),
    ("AAAA", 0.0),
    ("TTTT", 0.0),
    ("ATGC", 50.0),
    ("ATGCATGC", 50.0),
])
def test_gc_content_parametrized(seq, expected):
    assert gc_content(seq) == expected
'''

print(test_code)

### Running Tests

In [ ]:
# pytest commands
commands = '''
# Run all tests
pytest

# Verbose output
pytest -v

# Run specific test file
pytest test_bio_utils.py

# Run specific test class
pytest test_bio_utils.py::TestGCContent

# Run specific test
pytest test_bio_utils.py::TestGCContent::test_basic

# With coverage report
pytest --cov=bio_utils --cov-report=html

# Stop at first failure
pytest -x
'''
print(commands)

---
## Part 2: Testing Best Practices for Bioinformatics

In [ ]:
# Fixtures: Reusable test data
fixture_example = '''
import pytest
from Bio import SeqIO
from io import StringIO

@pytest.fixture
def sample_fasta():
    """Provide a sample FASTA file."""
    fasta_data = """>gene1 | Homo sapiens
ATGCGATCGATCGATCG
>gene2 | Mus musculus
GCTAGCTAGCTAGCTAG
"""
    return StringIO(fasta_data)

@pytest.fixture
def sample_records(sample_fasta):
    """Parse FASTA into SeqRecord objects."""
    return list(SeqIO.parse(sample_fasta, "fasta"))

def test_record_count(sample_records):
    assert len(sample_records) == 2

def test_first_sequence(sample_records):
    assert str(sample_records[0].seq) == "ATGCGATCGATCGATCG"
'''
print(fixture_example)

In [ ]:
# Testing file I/O with tmp_path
file_io_tests = '''
import pytest
from pathlib import Path

def write_fasta(filepath, sequences):
    """Write sequences to FASTA file."""
    with open(filepath, 'w') as f:
        for name, seq in sequences.items():
            f.write(f">{name}\\n{seq}\\n")

def test_write_fasta(tmp_path):
    # tmp_path is a pytest fixture (auto-cleaned)
    output = tmp_path / "test.fasta"
    
    write_fasta(output, {"seq1": "ATGC", "seq2": "GCTA"})
    
    # Verify file was created
    assert output.exists()
    
    # Verify content
    content = output.read_text()
    assert ">seq1" in content
    assert "ATGC" in content
'''
print(file_io_tests)

---
## Part 3: GitHub Actions CI/CD

In [ ]:
# .github/workflows/tests.yml
github_actions = '''
name: Tests

on:
  push:
    branches: [main, master]
  pull_request:
    branches: [main, master]

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.9", "3.10", "3.11"]
    
    steps:
    - uses: actions/checkout@v4
    
    - name: Set up Python ${{ matrix.python-version }}
      uses: actions/setup-python@v5
      with:
        python-version: ${{ matrix.python-version }}
    
    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install pytest pytest-cov
        pip install -r requirements.txt
    
    - name: Run tests
      run: |
        pytest --cov=src --cov-report=xml
    
    - name: Upload coverage
      uses: codecov/codecov-action@v3
      with:
        file: coverage.xml
'''
print(github_actions)

In [ ]:
# Linting workflow
linting_workflow = '''
name: Lint

on: [push, pull_request]

jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
    - uses: actions/checkout@v4
    
    - name: Set up Python
      uses: actions/setup-python@v5
      with:
        python-version: "3.11"
    
    - name: Install linters
      run: |
        pip install ruff black isort mypy
    
    - name: Check formatting (black)
      run: black --check .
    
    - name: Check imports (isort)
      run: isort --check-only .
    
    - name: Lint (ruff)
      run: ruff check .
    
    - name: Type check (mypy)
      run: mypy src/
'''
print(linting_workflow)

---
## Part 4: Project Structure

In [ ]:
# Recommended project layout
project_structure = '''
my_bioinformatics_tool/
├── .github/
│   └── workflows/
│       ├── tests.yml
│       └── lint.yml
├── src/
│   └── my_tool/
│       ├── __init__.py
│       ├── io.py           # File parsing
│       ├── analysis.py     # Core algorithms
│       └── utils.py        # Helper functions
├── tests/
│   ├── conftest.py         # Shared fixtures
│   ├── test_io.py
│   ├── test_analysis.py
│   └── data/               # Test data files
│       ├── sample.fasta
│       └── expected.txt
├── pyproject.toml          # Modern Python config
├── requirements.txt
└── README.md
'''
print(project_structure)

## 🧬 Exercise: Write Tests

Write pytest tests for these functions:

In [ ]:
def find_motif(sequence, motif):
    """Find all positions of motif in sequence (1-based)."""
    positions = []
    start = 0
    while True:
        pos = sequence.find(motif, start)
        if pos == -1:
            break
        positions.append(pos + 1)  # 1-based
        start = pos + 1
    return positions

# Write tests:
# - Test basic motif finding
# - Test overlapping motifs (e.g., "AA" in "AAAA")
# - Test motif not found
# - Test empty sequence/motif

## Summary

**Testing bioinformatics code:**
- Use **pytest** for clean, readable tests
- Test edge cases (empty, N's, case sensitivity)
- Use fixtures for reusable test data
- Use tmp_path for file I/O tests

**CI/CD with GitHub Actions:**
- Run tests on every push/PR
- Test multiple Python versions
- Add linting (ruff, black, mypy)
- Report coverage